# Linear Models: Mathematical Foundations — Theory Notebook

> *"The method of least squares is the automobile of modern statistical analysis."* — Stigler

Interactive derivations covering OLS geometry, Ridge/Lasso regularisation,
Bayesian linear regression, logistic regression, SVMs, bias-variance tradeoff,
and deep learning connections (LoRA, NTK, probing classifiers).


In [ ]:
import numpy as np
import scipy.linalg as la
from scipy import stats
from scipy.optimize import minimize

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
        HAS_SNS = True
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
        HAS_SNS = False
    mpl.rcParams.update({
        'figure.figsize': (10, 6), 'figure.dpi': 110,
        'font.size': 12, 'axes.titlesize': 14,
        'axes.labelsize': 12, 'legend.fontsize': 11,
        'axes.spines.top': False, 'axes.spines.right': False,
    })
    COLORS = {
        'primary': '#0077BB', 'secondary': '#EE7733',
        'tertiary': '#009988', 'error': '#CC3311',
        'neutral': '#555555', 'highlight': '#EE3377',
    }
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)
print('Setup complete.')


---

## 1  OLS: Geometry and Projection

The OLS estimator $\hat{\boldsymbol{\beta}} = (X^\top X)^{-1} X^\top \mathbf{y}$
projects the target vector $\mathbf{y}$ onto the **column space** of $X$.

Key properties to verify:
- $\hat{\mathbf{y}} = H\mathbf{y}$  where  $H = X(X^\top X)^{-1}X^\top$ (hat matrix)
- $H^2 = H$ (idempotent), $H^\top = H$ (symmetric)
- Residual $\mathbf{e} \perp \mathcal{C}(X)$: $X^\top\mathbf{e} = \mathbf{0}$


In [ ]:
# === 2.1 OLS Geometry ===

np.random.seed(42)
n, d = 50, 3
X = np.column_stack([np.ones(n), np.random.randn(n, d-1)])  # intercept + 2 features
beta_true = np.array([1.0, 2.0, -1.5])
y = X @ beta_true + 0.5 * np.random.randn(n)

# OLS solution
beta_hat = np.linalg.solve(X.T @ X, X.T @ y)
y_hat = X @ beta_hat
e = y - y_hat  # residuals

# Hat matrix
H = X @ np.linalg.inv(X.T @ X) @ X.T

print('OLS coefficients:          ', np.round(beta_hat, 4))
print('True coefficients:         ', beta_true)
print()
# Verify OLS properties
print(f'PASS Residuals orthogonal to X: max|X^T e| = {np.max(np.abs(X.T @ e)):.2e}')
print(f'PASS H is idempotent:           max|H^2-H| = {np.max(np.abs(H@H - H)):.2e}')
print(f'PASS H is symmetric:            max|H-H^T| = {np.max(np.abs(H - H.T)):.2e}')
print(f'PASS rank(H) = d:               trace(H)   = {np.trace(H):.6f}  (expect {d})')
print(f'PASS y_hat = Hy:                max diff   = {np.max(np.abs(H@y - y_hat)):.2e}')
print()
# Leverage scores
h_diag = np.diag(H)
print(f'Leverage scores: min={h_diag.min():.4f}, max={h_diag.max():.4f}, sum={h_diag.sum():.4f}')


In [ ]:
# === 2.2 Visualise Projection and Leverage ===

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # (a) Fitted vs actual
    axes[0].scatter(y_hat, y, alpha=0.6, color=COLORS['primary'], s=40)
    lims = [min(y.min(), y_hat.min())-0.3, max(y.max(), y_hat.max())+0.3]
    axes[0].plot(lims, lims, color=COLORS['neutral'], ls='--', lw=1.5)
    axes[0].set_xlabel('Fitted values $\\hat{y}$')
    axes[0].set_ylabel('Observed $y$')
    axes[0].set_title('Fitted vs Actual')

    # (b) Residuals vs fitted
    axes[1].scatter(y_hat, e, alpha=0.6, color=COLORS['secondary'], s=40)
    axes[1].axhline(0, color=COLORS['neutral'], ls='--', lw=1.5)
    axes[1].set_xlabel('Fitted values $\\hat{y}$')
    axes[1].set_ylabel('Residuals $e$')
    axes[1].set_title('Residuals vs Fitted')

    # (c) Leverage scores
    axes[2].stem(range(n), h_diag, linefmt=COLORS['tertiary'],
                 markerfmt='o', basefmt='k-')
    axes[2].axhline(2*d/n, color=COLORS['error'], ls='--', lw=1.5,
                    label=f'2d/n = {2*d/n:.3f} (threshold)')
    axes[2].set_xlabel('Observation index')
    axes[2].set_ylabel('Leverage $h_{ii}$')
    axes[2].set_title('Leverage Scores')
    axes[2].legend()
    fig.tight_layout()
    plt.suptitle('OLS Diagnostics: Projection and Leverage', y=1.02,
                 fontsize=14, fontweight='bold')
    plt.show()

    n_high_leverage = (h_diag > 2*d/n).sum()
    print(f'High-leverage points (h_ii > 2d/n={2*d/n:.3f}): {n_high_leverage}')


---

## 2  SVD-Based OLS and Condition Number

Using $X = U\Sigma V^\top$:
$$\hat{\boldsymbol{\beta}} = V\Sigma^{-1}U^\top \mathbf{y}, \quad H = UU^\top$$

The condition number $\kappa = \sigma_1/\sigma_d$ governs numerical stability.


In [ ]:
# === 2.4 SVD-Based OLS ===

U, s, Vt = np.linalg.svd(X, full_matrices=False)

# SVD-based OLS
beta_svd = Vt.T @ np.diag(1/s) @ U.T @ y

print('SVD beta:   ', np.round(beta_svd, 6))
print('Direct beta:', np.round(beta_hat, 6))
print(f'Max difference: {np.max(np.abs(beta_svd - beta_hat)):.2e}')
print()
print(f'Singular values of X: {np.round(s, 4)}')
print(f'Condition number kappa = sigma_1/sigma_d = {s[0]/s[-1]:.2f}')

# Demonstrate ill-conditioning
print("\n--- Effect of near-multicollinearity ---")
for noise_level in [0.1, 0.01, 0.001]:
    X_ill = np.column_stack([X[:, 0], X[:, 1], X[:, 1] + noise_level*np.random.randn(n)])
    _, s_ill, _ = np.linalg.svd(X_ill, full_matrices=False)
    kappa = s_ill[0] / s_ill[-1]
    beta_ill = np.linalg.lstsq(X_ill, y, rcond=None)[0]
    print(f'  noise={noise_level:.3f}: kappa={kappa:.1f}, ||beta||={np.linalg.norm(beta_ill):.2f}')


---

## 3  Ridge Regression: SVD Shrinkage

Ridge applies shrinkage factor $\sigma_j^2/(\sigma_j^2+\lambda)$ to each SVD component:
$$\hat{\boldsymbol{\beta}}_\lambda = \sum_{j=1}^d \frac{\sigma_j}{\sigma_j^2+\lambda}(\mathbf{u}_j^\top\mathbf{y})\mathbf{v}_j$$

Effective degrees of freedom: $\operatorname{df}(\lambda) = \sum_j \sigma_j^2/(\sigma_j^2+\lambda)$


In [ ]:
# === 4.2 Ridge SVD Shrinkage ===

def ridge_svd(X, y, lam):
    """Ridge regression via SVD."""
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    d_factors = s / (s**2 + lam)   # shrinkage
    beta = Vt.T @ np.diag(d_factors) @ U.T @ y
    df = np.sum(s**2 / (s**2 + lam))
    return beta, df

lambdas = np.logspace(-3, 4, 200)
paths = np.array([ridge_svd(X, y, lam)[0] for lam in lambdas])
dfs   = np.array([ridge_svd(X, y, lam)[1] for lam in lambdas])

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Ridge path
    for j in range(d):
        axes[0].semilogx(lambdas, paths[:, j],
                         label=f'$\\beta_{j+1}$', lw=2)
    axes[0].axhline(0, color=COLORS['neutral'], ls=':', lw=1)
    axes[0].set_xlabel('Regularisation $\\lambda$')
    axes[0].set_ylabel('Coefficient value')
    axes[0].set_title('Ridge Coefficient Path')
    axes[0].legend()

    # Effective df vs lambda
    axes[1].semilogx(lambdas, dfs, color=COLORS['secondary'], lw=2)
    axes[1].axhline(d, color=COLORS['neutral'], ls='--', lw=1, label=f'df=d={d} (OLS)')
    axes[1].axhline(0, color=COLORS['neutral'], ls=':', lw=1, label='df=0 (null)')
    axes[1].set_xlabel('Regularisation $\\lambda$')
    axes[1].set_ylabel('Effective df$(\\lambda)$')
    axes[1].set_title('Effective Degrees of Freedom')
    axes[1].legend()
    fig.tight_layout()
    plt.suptitle('Ridge Regression: Shrinkage and Degrees of Freedom',
                 y=1.01, fontsize=13, fontweight='bold')
    plt.show()

# Verify MAP interpretation
sigma2, tau2 = 0.25, 1.0
lam_bayes = sigma2 / tau2
beta_ridge, _ = ridge_svd(X, y, lam_bayes)
beta_map = np.linalg.solve(X.T@X + lam_bayes*np.eye(d), X.T@y)
print(f'Ridge SVD vs MAP: max diff = {np.max(np.abs(beta_ridge - beta_map)):.2e}  (should be ~0)')


In [ ]:
# === 4.4 Ridge Cross-Validation ===

def cv_error(X, y, lam, k=5):
    """k-fold CV error for ridge regression."""
    n = len(y)
    idx = np.arange(n)
    np.random.shuffle(idx)
    folds = np.array_split(idx, k)
    errors = []
    for fold in folds:
        train = np.concatenate([f for f in folds if f is not fold])
        X_tr, y_tr = X[train], y[train]
        X_val, y_val = X[fold], y[fold]
        beta, _ = ridge_svd(X_tr, y_tr, lam)
        errors.append(np.mean((y_val - X_val@beta)**2))
    return np.mean(errors)

np.random.seed(42)
lambdas_cv = np.logspace(-3, 3, 50)
cv_errors = [cv_error(X, y, lam) for lam in lambdas_cv]
best_lam = lambdas_cv[np.argmin(cv_errors)]

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.semilogx(lambdas_cv, cv_errors, color=COLORS['primary'], lw=2)
    ax.axvline(best_lam, color=COLORS['error'], ls='--', lw=1.5,
               label=f'Best $\\lambda$={best_lam:.4f}')
    ax.set_xlabel('Regularisation $\\lambda$')
    ax.set_ylabel('5-fold CV MSE')
    ax.set_title('Ridge: Cross-Validation Error vs $\\lambda$')
    ax.legend()
    fig.tight_layout()
    plt.show()

print(f'Best lambda (CV): {best_lam:.4f}')
beta_best, df_best = ridge_svd(X, y, best_lam)
print(f'Coefficients at best lambda: {np.round(beta_best, 4)}')
print(f'Effective df: {df_best:.2f}')


---

## 4  Lasso: Coordinate Descent and Sparsity

The soft-thresholding operator $S_\lambda(z) = \operatorname{sign}(z)\max(|z|-\lambda, 0)$
gives exact zeros — the key to Lasso sparsity.

Coordinate descent cycles through features, applying $S_\lambda$ to each partial OLS estimate.


In [ ]:
# === 5.3 Lasso Coordinate Descent ===

def soft_threshold(z, lam):
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0)

def lasso_cd(X, y, lam, tol=1e-8, max_iter=2000):
    """Cyclic coordinate descent for Lasso."""
    n, d = X.shape
    beta = np.zeros(d)
    r = y.copy()  # residual = y - X@beta
    # Precompute column norms^2
    col_norms2 = np.sum(X**2, axis=0)

    for iteration in range(max_iter):
        beta_old = beta.copy()
        for j in range(d):
            r += X[:, j] * beta[j]               # restore residual for coord j
            z_j = X[:, j] @ r / n                 # partial OLS estimate
            beta[j] = soft_threshold(z_j, lam)    # update
            r -= X[:, j] * beta[j]               # update residual
        if np.max(np.abs(beta - beta_old)) < tol:
            break
    return beta

# Sparse true model
np.random.seed(0)
n, d = 100, 20
X_l = np.random.randn(n, d)
beta_sparse = np.zeros(d)
beta_sparse[[0, 3, 7]] = [3.0, -2.0, 1.5]  # only 3 nonzero
y_l = X_l @ beta_sparse + 0.5 * np.random.randn(n)

# Fit Lasso
lam_lasso = 0.2
beta_lasso = lasso_cd(X_l, y_l, lam_lasso)

print(f'True nonzero: indices={np.where(beta_sparse!=0)[0]}, values={beta_sparse[beta_sparse!=0]}')
print(f'Lasso nonzero: indices={np.where(beta_lasso!=0)[0]}, values={np.round(beta_lasso[beta_lasso!=0],3)}')
print(f'OLS nonzero: {(np.linalg.lstsq(X_l, y_l, rcond=None)[0] != 0).sum()} (all d={d} nonzero)')


In [ ]:
# === 5.3 Lasso Regularisation Path ===

lambdas_l = np.linspace(0.5, 0.001, 100)
lasso_paths = np.array([lasso_cd(X_l, y_l, lam) for lam in lambdas_l])

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Lasso path
    for j in range(d):
        color = COLORS['error'] if beta_sparse[j] != 0 else COLORS['neutral']
        alpha_val = 0.9 if beta_sparse[j] != 0 else 0.3
        lw = 2 if beta_sparse[j] != 0 else 0.8
        axes[0].plot(lambdas_l, lasso_paths[:, j],
                     color=color, alpha=alpha_val, lw=lw)
    axes[0].axvline(lam_lasso, color=COLORS['highlight'], ls='--', lw=1.5,
                    label=f'$\\lambda$={lam_lasso}')
    axes[0].set_xlabel('$\\lambda$'); axes[0].set_ylabel('Coefficient value')
    axes[0].set_title('Lasso Path  (red = true nonzero, grey = true zero)')
    axes[0].legend()

    # Sparsity vs lambda
    n_nonzero = (lasso_paths != 0).sum(axis=1)
    axes[1].plot(lambdas_l, n_nonzero, color=COLORS['tertiary'], lw=2)
    axes[1].set_xlabel('$\\lambda$'); axes[1].set_ylabel('Number of nonzero coefficients')
    axes[1].set_title('Lasso Sparsity vs $\\lambda$')
    fig.tight_layout()
    plt.suptitle('Lasso Regularisation Path and Sparsity',
                 y=1.01, fontsize=13, fontweight='bold')
    plt.show()


In [ ]:
# === 5.1 L1 vs L2 Ball Geometry ===

if HAS_MPL:
    theta = np.linspace(0, 2*np.pi, 400)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, title, ball_x, ball_y in [
        (axes[0], 'L1 ball (Lasso)',
         np.array([1,0,-1,0,1]), np.array([0,1,0,-1,0])),
        (axes[1], 'L2 ball (Ridge)',
         np.cos(theta), np.sin(theta)),
    ]:
        ax.fill(ball_x, ball_y, alpha=0.25, color=COLORS['tertiary'])
        ax.plot(ball_x, ball_y, color=COLORS['tertiary'], lw=2)

        # Elliptical contours of OLS objective centered at (0.8, 0.6)
        cx, cy = 0.8, 0.6
        for r in [0.3, 0.5, 0.7, 0.9]:
            ex = cx + r * np.cos(theta) * 0.8
            ey = cy + r * np.sin(theta) * 0.6
            ax.plot(ex, ey, color=COLORS['primary'], alpha=0.4, lw=1)

        ax.scatter([cx], [cy], color=COLORS['primary'], s=100, zorder=5, label='OLS min')
        ax.set_aspect('equal'); ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
        ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
        ax.set_xlabel('$\\beta_1$'); ax.set_ylabel('$\\beta_2$')
        ax.set_title(title); ax.legend()

    axes[0].annotate('Contact at corner\n→ exact zero!',
                     xy=(0, 1.02), xytext=(0.5, 1.3),
                     arrowprops=dict(arrowstyle='->', color=COLORS['error']),
                     fontsize=11, color=COLORS['error'])
    plt.suptitle('Geometry of Regularisation: L1 vs L2 Constraint Regions',
                 fontsize=13, fontweight='bold')
    fig.tight_layout()
    plt.show()


---

## 5  Bayesian Linear Regression

With Gaussian prior $\boldsymbol{\beta}\sim\mathcal{N}(\boldsymbol{\mu}_0, \Sigma_0)$:
$$\Sigma_n^{-1} = \Sigma_0^{-1} + \frac{1}{\sigma^2}X^\top X, \quad
\boldsymbol{\mu}_n = \Sigma_n\!\left(\Sigma_0^{-1}\boldsymbol{\mu}_0 + \frac{1}{\sigma^2}X^\top\mathbf{y}\right)$$

Predictive variance = epistemic ($\mathbf{x}_*^\top\Sigma_n\mathbf{x}_*$) + aleatoric ($\sigma^2$).


In [ ]:
# === 7.1-7.3 Bayesian Linear Regression ===

def bayes_lr(X, y, sigma2, mu0, Sigma0_inv):
    """Compute Bayesian linear regression posterior."""
    Sigma_n_inv = Sigma0_inv + (1/sigma2) * X.T @ X
    Sigma_n = np.linalg.inv(Sigma_n_inv)
    mu_n = Sigma_n @ (Sigma0_inv @ mu0 + (1/sigma2) * X.T @ y)
    return mu_n, Sigma_n

def predictive(x_star, mu_n, Sigma_n, sigma2):
    """Bayesian predictive mean and variance."""
    mean = x_star @ mu_n
    var  = x_star @ Sigma_n @ x_star + sigma2
    return mean, var

# 1-D example: y = beta*x + noise, with x-intercept absorbed
np.random.seed(42)
x_1d = np.linspace(-3, 3, 8)
beta_1d = 1.5
sigma2 = 0.5
y_1d = beta_1d * x_1d + np.sqrt(sigma2) * np.random.randn(len(x_1d))
X_1d = x_1d.reshape(-1, 1)  # no intercept for simplicity

# Prior: N(0, 2^2 I)
mu0 = np.array([0.0])
Sigma0_inv = (1/4.0) * np.eye(1)

x_test = np.linspace(-4, 4, 200).reshape(-1, 1)
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax_idx, n_obs in enumerate([2, 5, len(x_1d)]):
        X_n = X_1d[:n_obs]; y_n = y_1d[:n_obs]
        mu_n, Sigma_n = bayes_lr(X_n, y_n, sigma2, mu0, Sigma0_inv)

        means = np.array([predictive(xt, mu_n, Sigma_n, sigma2)[0] for xt in x_test])
        stds  = np.array([np.sqrt(predictive(xt, mu_n, Sigma_n, sigma2)[1]) for xt in x_test])

        ax = axes[ax_idx]
        ax.plot(x_test, beta_1d * x_test, color=COLORS['neutral'],
                ls='--', lw=1.5, label='True')
        ax.plot(x_test.ravel(), means, color=COLORS['primary'], lw=2, label='Posterior mean')
        ax.fill_between(x_test.ravel(), means - 2*stds, means + 2*stds,
                        color=COLORS['primary'], alpha=0.15, label='95% PI')
        ax.scatter(x_1d[:n_obs], y_1d[:n_obs], color=COLORS['error'],
                   s=60, zorder=5, label='Data')
        ax.set_xlabel('x'); ax.set_ylabel('y')
        ax.set_title(f'n={n_obs} observations\n$\\mu_n$={mu_n[0]:.3f}')
        ax.legend(fontsize=9)
    fig.tight_layout()
    plt.suptitle('Bayesian Linear Regression: Epistemic Uncertainty Decreases with Data',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.show()

mu_final, Sigma_final = bayes_lr(X_1d, y_1d, sigma2, mu0, Sigma0_inv)
print(f'Posterior mean (all data):     beta = {mu_final[0]:.4f}')
print(f'Posterior std:                       {np.sqrt(Sigma_final[0,0]):.4f}')
print(f'Ridge estimate (lambda=sigma2/4): {ridge_svd(X_1d, y_1d, sigma2/4)[0][0]:.4f}')
print('Posterior mean = Ridge MAP estimate (as expected)')


---

## 6  Logistic Regression: Gradient, Hessian, and IRLS

The cross-entropy gradient $\nabla_\mathbf{w}\mathcal{L} = \frac{1}{n}X^\top(\mathbf{p}-\mathbf{y})$
has the same structure as the OLS gradient — residuals dotted with features.

The Hessian $H = \frac{1}{n}X^\top DX$ with $D=\operatorname{diag}(p_i(1-p_i))$ is PSD — global convexity.


In [1]:
# === 8.1-8.2 Logistic Regression: Gradient Descent and IRLS ===

def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def logistic_loss_grad(w, X, y):
    n = len(y)
    p = sigmoid(X @ w)
    loss = -np.mean(y * np.log(p + 1e-15) + (1-y) * np.log(1-p + 1e-15))
    grad = X.T @ (p - y) / n
    return loss, grad

def irls_step(w, X, y):
    """One Newton (IRLS) step for logistic regression."""
    n = len(y)
    p = sigmoid(X @ w)
    d_diag = p * (1 - p)
    # Adjusted response
    z = X @ w + (y - p) / (d_diag + 1e-15)
    # Weighted least squares
    W_half = np.sqrt(d_diag)
    XW = X * W_half[:, None]
    zW = z * W_half
    w_new, _, _, _ = np.linalg.lstsq(XW, zW, rcond=None)
    return w_new

# Generate 2-class dataset
np.random.seed(42)
n_cls = 200
X_cls = np.column_stack([
    np.ones(n_cls),
    np.random.randn(n_cls, 2)
])
w_true = np.array([0.5, 1.5, -1.0])
y_cls = (sigmoid(X_cls @ w_true) > 0.5).astype(float)

# Gradient descent
w_gd = np.zeros(3); losses_gd = []
for _ in range(500):
    loss, grad = logistic_loss_grad(w_gd, X_cls, y_cls)
    losses_gd.append(loss)
    w_gd -= 0.1 * grad

# IRLS (Newton)
w_irls = np.zeros(3); losses_irls = []
for _ in range(20):
    loss, _ = logistic_loss_grad(w_irls, X_cls, y_cls)
    losses_irls.append(loss)
    w_irls = irls_step(w_irls, X_cls, y_cls)

print('Gradient descent (500 steps): w =', np.round(w_gd, 4))
print('IRLS (20 steps):              w =', np.round(w_irls, 4))
print('True weights:                 w =', w_true)

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].semilogy(losses_gd, color=COLORS['primary'], lw=2, label='GD (500 steps)')
    axes[0].semilogy(np.arange(len(losses_irls)), losses_irls,
                     color=COLORS['error'], lw=2, marker='o', ms=6, label='IRLS (20 steps)')
    axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Loss (log scale)')
    axes[0].set_title('GD vs IRLS Convergence'); axes[0].legend()

    # Decision boundary
    x1_range = np.linspace(X_cls[:,1].min()-0.5, X_cls[:,1].max()+0.5, 100)
    x2_range = -(w_irls[0] + w_irls[1]*x1_range) / w_irls[2]
    axes[1].scatter(X_cls[y_cls==0,1], X_cls[y_cls==0,2],
                    color=COLORS['primary'], alpha=0.5, label='Class 0')
    axes[1].scatter(X_cls[y_cls==1,1], X_cls[y_cls==1,2],
                    color=COLORS['error'], alpha=0.5, label='Class 1')
    axes[1].plot(x1_range, x2_range, color=COLORS['neutral'], lw=2, label='Decision boundary')
    axes[1].set_xlabel('$x_1$'); axes[1].set_ylabel('$x_2$')
    axes[1].set_title('Logistic Regression Decision Boundary'); axes[1].legend()
    fig.tight_layout()
    plt.suptitle('Logistic Regression: GD vs IRLS and Decision Boundary',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.show()


NameError: name 'np' is not defined

---

## 7  LDA vs Logistic Regression

LDA assumes class-conditional Gaussians with **shared covariance** $\Sigma$.
The decision rule $\delta_k(\mathbf{x}) = \mathbf{x}^\top\hat{\Sigma}^{-1}\hat{\boldsymbol{\mu}}_k - \frac{1}{2}\hat{\boldsymbol{\mu}}_k^\top\hat{\Sigma}^{-1}\hat{\boldsymbol{\mu}}_k + \log\hat{\pi}_k$
is **linear in $\mathbf{x}$** — same decision boundary as logistic regression.

Key difference: LDA estimates the boundary *via* the generative model; logistic regression estimates it *directly*.


In [ ]:
# === 9.2 LDA vs Logistic Regression ===

def lda_fit(X, y):
    """Fit LDA and return discriminant function parameters."""
    classes = np.unique(y)
    K = len(classes)
    n, d = X.shape
    mus = np.array([X[y==k].mean(axis=0) for k in classes])
    pis = np.array([np.mean(y==k) for k in classes])
    # Pooled within-class covariance
    Sigma = sum([(X[y==k] - mus[k]).T @ (X[y==k] - mus[k]) for k in range(K)]) / (n - K)
    Sigma_inv = np.linalg.inv(Sigma + 1e-6 * np.eye(d))
    return mus, pis, Sigma_inv

def lda_predict(X, mus, pis, Sigma_inv):
    scores = np.array([
        X @ Sigma_inv @ mus[k] - 0.5 * mus[k] @ Sigma_inv @ mus[k] + np.log(pis[k])
        for k in range(len(mus))
    ]).T
    return scores.argmax(axis=1)

# Compare with different n scenarios (Ng & Jordan analysis)
np.random.seed(42)
results = []
for n_small in [20, 50, 200, 1000]:
    # Two Gaussians with shared covariance
    X_ng = np.vstack([
        np.random.multivariate_normal([0,0], [[1,0.8],[0.8,1]], n_small//2),
        np.random.multivariate_normal([2,2], [[1,0.8],[0.8,1]], n_small//2),
    ])
    y_ng = np.array([0]*(n_small//2) + [1]*(n_small//2))

    # Test data
    X_test_ng = np.vstack([
        np.random.multivariate_normal([0,0], [[1,0.8],[0.8,1]], 500),
        np.random.multivariate_normal([2,2], [[1,0.8],[0.8,1]], 500),
    ])
    y_test_ng = np.array([0]*500 + [1]*500)

    # LDA
    mus, pis, Sigma_inv = lda_fit(X_ng, y_ng)
    acc_lda = np.mean(lda_predict(X_test_ng, mus, pis, Sigma_inv) == y_test_ng)

    # Logistic regression (via scipy)
    X_ng_aug = np.column_stack([np.ones(n_small), X_ng])
    w_lg = np.zeros(3)
    for _ in range(300):
        _, grad = logistic_loss_grad(w_lg, X_ng_aug, y_ng)
        w_lg -= 0.05 * grad
    X_test_aug = np.column_stack([np.ones(1000), X_test_ng])
    acc_log = np.mean((sigmoid(X_test_aug @ w_lg) > 0.5) == y_test_ng)

    results.append((n_small, acc_lda, acc_log))

print(f'  n     LDA     Logistic   Winner')
print('-' * 40)
for n_s, a_lda, a_log in results:
    winner = 'LDA' if a_lda > a_log else 'Logistic'
    print(f' {n_s:4d}  {a_lda:.3f}   {a_log:.3f}     {winner}')
print('\nExpected: LDA wins for small n (generative advantage),')
print('Logistic wins for large n (discriminative advantage).')


---

## 8  Support Vector Machines: Margin and Duality

Hard-margin SVM primal: $\min \frac{1}{2}\lVert\mathbf{w}\rVert^2$ s.t. $y^{(i)}(\mathbf{w}^\top\mathbf{x}^{(i)}+b)\geq 1$.

Dual: $\max_{\boldsymbol{\alpha}} \sum\alpha_i - \frac{1}{2}\sum_{i,j}\alpha_i\alpha_j y^{(i)}y^{(j)}(\mathbf{x}^{(i)})^\top\mathbf{x}^{(j)}$ s.t. $\alpha_i\geq 0$, $\sum\alpha_i y^{(i)}=0$.

Support vectors: the training points with $\alpha_i > 0$ (on the margin boundary).


In [ ]:
# === 10.1-10.2 SVM: Margin Maximisation ===

import numpy as np
try:
    import matplotlib.pyplot as plt
    COLORS = {
        'primary': '#0077BB', 'secondary': '#EE7733',
        'tertiary': '#009988', 'error': '#CC3311',
        'neutral': '#555555', 'highlight': '#EE3377',
    }
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def svm_hinge_loss_grad(w, b, X, y, C=1.0):
    """Soft-margin SVM hinge loss and gradient (for GD implementation)."""
    n = len(y)
    margins = y * (X @ w + b)
    loss = 0.5 * np.dot(w, w) + C * np.mean(np.maximum(0, 1 - margins))
    # Gradient
    mask = margins < 1
    dw = w - C * (X[mask].T @ y[mask]) / n
    db = -C * y[mask].mean() if mask.any() else 0.0
    return loss, dw, db

# Generate linearly separable data
np.random.seed(7)
n_svm = 60
X_svm = np.vstack([
    np.random.randn(n_svm//2, 2) + [-1.5, 0],
    np.random.randn(n_svm//2, 2) + [1.5, 0],
])
y_svm = np.array([-1]*(n_svm//2) + [1]*(n_svm//2), dtype=float)

# Train with projected gradient descent
w_svm = np.zeros(2); b_svm = 0.0
for _ in range(2000):
    _, dw, db = svm_hinge_loss_grad(w_svm, b_svm, X_svm, y_svm, C=1.0)
    w_svm -= 0.01 * dw; b_svm -= 0.01 * db

margin = 2.0 / np.linalg.norm(w_svm)
print(f'Weight vector: {np.round(w_svm, 4)}')
print(f'Margin: {margin:.4f}')

# Support vectors (on or inside margin)
margins_train = y_svm * (X_svm @ w_svm + b_svm)
sv_mask = margins_train <= 1.01
print(f'Support vectors: {sv_mask.sum()} out of {n_svm}')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 6))
    # Data
    ax.scatter(X_svm[y_svm==-1,0], X_svm[y_svm==-1,1],
               color=COLORS['primary'], label='Class -1', s=50, alpha=0.7)
    ax.scatter(X_svm[y_svm==1,0], X_svm[y_svm==1,1],
               color=COLORS['error'], label='Class +1', s=50, alpha=0.7)
    ax.scatter(X_svm[sv_mask,0], X_svm[sv_mask,1],
               facecolors='none', edgecolors=COLORS['neutral'], s=120, lw=2, label='Support vectors')
    # Decision boundary and margins
    x1_r = np.linspace(X_svm[:,0].min()-0.5, X_svm[:,0].max()+0.5, 200)
    for offset, ls, lbl in [(0, '-', 'Decision boundary'), (1, '--', 'Margin'), (-1, '--', None)]:
        x2_r = -(w_svm[0]*x1_r + b_svm - offset) / w_svm[1]
        ax.plot(x1_r, x2_r, color=COLORS['neutral'], ls=ls, lw=1.5,
                label=lbl if lbl else '')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(f'SVM Decision Boundary  (margin = {margin:.3f})')
    ax.legend(); fig.tight_layout()
    plt.show()


In [ ]:
# === 10.4 Kernel SVM: RBF Kernel ===

def rbf_kernel(X1, X2, gamma=1.0):
    """K[i,j] = exp(-gamma * ||x1_i - x2_j||^2)."""
    diff = X1[:, None, :] - X2[None, :, :]
    return np.exp(-gamma * np.sum(diff**2, axis=-1))

# Visualise kernel matrices for different kernels
np.random.seed(42)
X_k = np.random.randn(30, 2)  # 30 points in 2D
kernels = {
    'Linear $x^\\top z$': lambda X: X @ X.T,
    'Poly $(x^\\top z + 1)^2$': lambda X: (X @ X.T + 1)**2,
    'RBF $e^{-||x-z||^2}$': lambda X: rbf_kernel(X, X, gamma=1.0),
}

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (name, kern_fn) in zip(axes, kernels.items()):
        K = kern_fn(X_k)
        im = ax.imshow(K, cmap='viridis', aspect='auto')
        plt.colorbar(im, ax=ax, fraction=0.046)
        ax.set_title(name); ax.set_xlabel('Sample j'); ax.set_ylabel('Sample i')
    plt.suptitle('Kernel Matrices: Linear, Polynomial, RBF',
                 fontsize=13, fontweight='bold')
    fig.tight_layout(); plt.show()

# Verify Mercer's condition: all eigenvalues >= 0
for name, kern_fn in kernels.items():
    K = kern_fn(X_k)
    eigvals = np.linalg.eigvalsh(K)
    print(f'{name}: min eigenvalue = {eigvals.min():.6f}  '
          f'({"PSD" if eigvals.min() >= -1e-8 else "NOT PSD"})')


---

## 9  Bias-Variance Tradeoff and Double Descent

$$\mathbb{E}[(y-\hat{f})^2] = \text{Bias}^2 + \text{Variance} + \sigma^2$$

For Ridge: bias$^2$ increases with $\lambda$, variance decreases. Optimal $\lambda^*$ minimises MSE.

**Double descent:** at the interpolation threshold $d = n$, test error spikes,
then decreases again as $d/n \to \infty$.


In [ ]:
# === 11.1 Bias-Variance Decomposition for Ridge ===

np.random.seed(42)
n_bv, d_bv = 40, 5
sigma2_bv = 0.5

# True design matrix (fixed X)
X_bv = np.random.randn(n_bv, d_bv)
beta_bv = np.array([2., -1., 1.5, 0., 0.5])

# Analytical bias and variance for Ridge
U_bv, s_bv, Vt_bv = np.linalg.svd(X_bv, full_matrices=False)
x_star = np.random.randn(d_bv)  # test point

lambdas_bv = np.logspace(-3, 4, 200)
biases2, variances, mses = [], [], []

for lam in lambdas_bv:
    # Ridge beta at this lambda
    shrinkage = s_bv / (s_bv**2 + lam)
    beta_ridge = Vt_bv.T @ np.diag(shrinkage) @ U_bv.T @ (X_bv @ beta_bv)

    # Bias at x_star
    bias2 = (x_star @ beta_ridge - x_star @ beta_bv)**2

    # Variance at x_star
    H_lam = X_bv @ np.diag(shrinkage**2) @ np.diag(s_bv**2) @ X_bv.T
    # Var(hat_y(x*)) = sigma^2 * x*^T (X^TX + lI)^{-2} X^TX x*
    A = Vt_bv.T @ np.diag(shrinkage**2 * s_bv**2) @ Vt_bv
    var = sigma2_bv * x_star @ A @ x_star

    biases2.append(bias2)
    variances.append(var)
    mses.append(bias2 + var + sigma2_bv)

opt_idx = np.argmin(mses)
print(f'Optimal lambda: {lambdas_bv[opt_idx]:.4f}')
print(f'Min MSE: {mses[opt_idx]:.4f}')
print(f'At optimal lambda: Bias^2={biases2[opt_idx]:.4f}, Var={variances[opt_idx]:.4f}')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.semilogx(lambdas_bv, biases2, color=COLORS['error'], lw=2, label='Bias$^2$')
    ax.semilogx(lambdas_bv, variances, color=COLORS['tertiary'], lw=2, label='Variance')
    ax.semilogx(lambdas_bv, mses, color=COLORS['primary'], lw=2.5, label='MSE = Bias²+Var+σ²')
    ax.axvline(lambdas_bv[opt_idx], color=COLORS['neutral'], ls='--', lw=1.5,
               label=f'Optimal $\\lambda$={lambdas_bv[opt_idx]:.4f}')
    ax.set_xlabel('$\\lambda$'); ax.set_ylabel('Error component')
    ax.set_title('Bias-Variance Tradeoff for Ridge Regression')
    ax.legend(); fig.tight_layout(); plt.show()


In [ ]:
# === 11.2 Double Descent Phenomenon ===

np.random.seed(42)
n_dd = 50        # fixed sample size
d_values = list(range(2, 150, 3))   # vary model complexity
sigma2_dd = 0.3
n_trials = 20

test_mses = []
for d_dd in d_values:
    trial_mses = []
    for _ in range(n_trials):
        beta_true_dd = np.zeros(d_dd)
        beta_true_dd[:5] = [2., -1., 1.5, 0.8, -0.5]  # only 5 nonzero
        X_dd = np.random.randn(n_dd, d_dd) / np.sqrt(d_dd)
        y_dd = X_dd @ beta_true_dd + np.sqrt(sigma2_dd) * np.random.randn(n_dd)
        X_test_dd = np.random.randn(200, d_dd) / np.sqrt(d_dd)
        y_test_dd = X_test_dd @ beta_true_dd + np.sqrt(sigma2_dd) * np.random.randn(200)
        # Minimum norm solution
        beta_dd, _, _, _ = np.linalg.lstsq(X_dd, y_dd, rcond=None)
        mse = np.mean((y_test_dd - X_test_dd @ beta_dd)**2)
        trial_mses.append(min(mse, 20))  # cap for display
    test_mses.append(np.median(trial_mses))

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(d_values, test_mses, color=COLORS['primary'], lw=2)
    ax.axvline(n_dd, color=COLORS['error'], ls='--', lw=2,
               label=f'Interpolation threshold (d=n={n_dd})')
    ax.axhline(sigma2_dd, color=COLORS['neutral'], ls=':', lw=1.5, label=f'Noise floor σ²={sigma2_dd}')
    ax.set_xlabel('Number of parameters $d$')
    ax.set_ylabel('Test MSE')
    ax.set_title('Double Descent: Test Error vs Model Size  (n=50 fixed)')
    ax.legend(); ax.set_ylim(0, 8)
    fig.tight_layout(); plt.show()
    print(f'Peak at d≈n={n_dd}: MSE={test_mses[d_values.index(min(d_values, key=lambda x: abs(x-n_dd)))]:.2f}')
    print('Second descent: error recovers as d >> n')


---

## 10  LoRA: Low-Rank Linear Regression

**LoRA** (Hu et al., 2022) fine-tunes $W = W_0 + BA$ with $B\in\mathbb{R}^{d\times r}$, $A\in\mathbb{R}^{r\times k}$, $r\ll\min(d,k)$.

This is equivalent to low-rank regression on the residuals $R = Y - XW_0$:
$$\min_{\Delta W} \frac{1}{2}\lVert R - X\Delta W\rVert_F^2 \quad \text{s.t.} \quad \operatorname{rank}(\Delta W) \leq r$$

The optimal rank-$r$ solution is given by the Eckart-Young theorem (truncated SVD of residuals).


In [ ]:
# === 13.2 LoRA: Low-Rank Adaptation ===

np.random.seed(42)
d_lora, k_lora, n_lora = 64, 64, 200

# Pretrained weight
W0 = np.random.randn(d_lora, k_lora) * 0.1

# Task data: target is a low-rank transformation from W0
r_true = 4  # true rank of adaptation
B_true = np.random.randn(d_lora, r_true)
A_true = np.random.randn(r_true, k_lora)
W_target = W0 + B_true @ A_true / r_true

X_lora = np.random.randn(n_lora, k_lora)  # input features
Y_lora = X_lora @ W_target.T + 0.01 * np.random.randn(n_lora, d_lora)  # outputs

# Compute residuals
R = Y_lora - X_lora @ W0.T  # (n, d)

# Optimal rank-r update (Eckart-Young)
# Delta_W_optimal = argmin ||R - X DeltaW.T||_F s.t. rank <= r
# This is low-rank OLS: compute DeltaW = (X^T X)^{-1} X^T R, then truncate SVD
DeltaW_full = np.linalg.lstsq(X_lora, R, rcond=None)[0].T  # (d, k)

U_dw, s_dw, Vt_dw = np.linalg.svd(DeltaW_full, full_matrices=False)

errs = []
for r_test in range(1, 20):
    DeltaW_r = U_dw[:, :r_test] @ np.diag(s_dw[:r_test]) @ Vt_dw[:r_test, :]
    W_approx = W0 + DeltaW_r
    err = np.mean((Y_lora - X_lora @ W_approx.T)**2)
    errs.append(err)

print(f'True adaptation rank: {r_true}')
print(f'Error at r=1: {errs[0]:.6f}')
print(f'Error at r={r_true}: {errs[r_true-1]:.6f}')
print(f'Error at r=8: {errs[7]:.6f}')

# Parameter counts
full_ft = d_lora * k_lora
for r_count in [4, 8, 16, 64]:
    lora_params = r_count * (d_lora + k_lora)
    print(f'r={r_count:3d}: LoRA params={lora_params:5d}  (full={full_ft}, ratio={lora_params/full_ft:.3f})')

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(range(1, 20), errs, color=COLORS['primary'], lw=2, marker='o', ms=5)
    axes[0].axvline(r_true, color=COLORS['error'], ls='--', lw=1.5,
                    label=f'True rank={r_true}')
    axes[0].set_xlabel('LoRA rank $r$'); axes[0].set_ylabel('Test MSE')
    axes[0].set_title('LoRA: Test MSE vs Rank'); axes[0].legend()
    axes[1].semilogy(range(1, len(s_dw)+1), s_dw,
                     color=COLORS['secondary'], lw=2, marker='s', ms=4)
    axes[1].axvline(r_true, color=COLORS['error'], ls='--', lw=1.5,
                    label=f'True rank={r_true}')
    axes[1].set_xlabel('Index'); axes[1].set_ylabel('Singular value $\\sigma_i$ (log)')
    axes[1].set_title('Singular Values of Optimal $\\Delta W$'); axes[1].legend()
    fig.tight_layout()
    plt.suptitle('LoRA = Low-Rank OLS on Residuals', fontsize=13, fontweight='bold', y=1.01)
    plt.show()


---

## 11  Neural Tangent Kernel (NTK)

For a 2-layer linear network $f(\mathbf{x}) = W_2 W_1 \mathbf{x}$ with parameters $\theta=(W_1,W_2)$:

$$\Theta(\mathbf{x},\mathbf{x}') = \nabla_\theta f(\mathbf{x})^\top \nabla_\theta f(\mathbf{x}')$$

Under gradient flow, the solution converges to kernel regression: $f_\infty = K_*^\top K^{-1}\mathbf{y}$.


In [ ]:
# === 13.4 Neural Tangent Kernel for Linear Network ===

np.random.seed(42)

# 2-layer linear network: f(x) = W2 @ W1 @ x
# theta = (W1, W2), d_in=2, h=4 (hidden), d_out=1
d_in, h_dim, d_out = 2, 8, 1

def ntk_linear_net(X1, X2, W1, W2):
    """
    NTK for f(x) = W2 @ W1 @ x.
    grad_theta f(x) = [W2.T ⊗ x (for W1), W1@x (for W2)]
    K[i,j] = grad_theta f(x_i)^T grad_theta f(x_j)
    """
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    for i, xi in enumerate(X1):
        gi_W1 = np.outer(W2.T, xi)  # dL/dW1 for unit output
        gi_W2 = W1 @ xi              # dL/dW2 for unit output
        gi = np.concatenate([gi_W1.ravel(), gi_W2.ravel()])
        for j, xj in enumerate(X2):
            gj_W1 = np.outer(W2.T, xj)
            gj_W2 = W1 @ xj
            gj = np.concatenate([gj_W1.ravel(), gj_W2.ravel()])
            K[i, j] = gi @ gj
    return K

# Training data
n_ntk = 20
X_ntk = np.random.randn(n_ntk, d_in)
beta_ntk = np.array([[1.5, -1.0]])  # true linear
y_ntk = (X_ntk @ beta_ntk.T).ravel() + 0.1 * np.random.randn(n_ntk)

# Wide random initialisation
scale = 1.0 / np.sqrt(h_dim)
W1_0 = np.random.randn(h_dim, d_in) * scale
W2_0 = np.random.randn(d_out, h_dim) * scale

# Compute NTK at initialisation
K_train = ntk_linear_net(X_ntk, X_ntk, W1_0, W2_0)

# Test points
X_test_ntk = np.random.randn(30, d_in)
K_test = ntk_linear_net(X_test_ntk, X_ntk, W1_0, W2_0)

# Kernel regression prediction
reg = 1e-4
alpha_ntk = np.linalg.solve(K_train + reg * np.eye(n_ntk), y_ntk)
y_pred_ntk = K_test @ alpha_ntk

# Compare with direct linear regression (should match in linear network)
y_pred_lr = X_test_ntk @ np.linalg.lstsq(X_ntk, y_ntk, rcond=None)[0]

corr = np.corrcoef(y_pred_ntk, y_pred_lr)[0, 1]
print(f'NTK kernel regression vs linear regression correlation: {corr:.4f}')
print(f'(Should be near 1 for linear network in lazy training regime)')
print()
print('NTK matrix eigenvalues (first 5):')
print(np.round(np.sort(np.linalg.eigvalsh(K_train))[::-1][:5], 4))


---

## 12  Attention as Linear Mixing

Attention output $\mathbf{o}_t = \sum_s A_{ts} W_V \mathbf{x}_s$ is a **weighted linear combination** of value vectors.

The $W_V W_O^\top$ product has rank $\leq d_k \ll d_{\text{model}}$ — multi-head attention applies multiple low-rank linear projections.


In [ ]:
# === 13.5 Attention as Linear Mixing ===

np.random.seed(42)
T = 8     # sequence length
d_model = 16
d_k = 4   # head dimension

# Weight matrices for one attention head
W_Q = np.random.randn(d_k, d_model) * 0.1
W_K = np.random.randn(d_k, d_model) * 0.1
W_V = np.random.randn(d_k, d_model) * 0.1
W_O = np.random.randn(d_model, d_k) * 0.1

# Input sequence
X_attn = np.random.randn(T, d_model)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=axis, keepdims=True)

# Scaled dot-product attention
Q = X_attn @ W_Q.T   # (T, d_k)
K = X_attn @ W_K.T   # (T, d_k)
V = X_attn @ W_V.T   # (T, d_k)
A = softmax(Q @ K.T / np.sqrt(d_k))  # (T, T)
output = A @ V @ W_O.T  # (T, d_model)

# OV circuit: W_O @ W_V = effective linear map
W_OV = W_O @ W_V   # (d_model, d_model)
U_ov, s_ov, Vt_ov = np.linalg.svd(W_OV)

print(f'Attention matrix A (shape {A.shape}):')
print(np.round(A, 3))
print(f'\nOV matrix rank: {np.linalg.matrix_rank(W_OV, tol=1e-6)} (max possible = d_k = {d_k})')
print(f'OV singular values: {np.round(s_ov[:6], 4)}')

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    im0 = axes[0].imshow(A, cmap='viridis', aspect='auto')
    plt.colorbar(im0, ax=axes[0], label='Attention weight')
    axes[0].set_title(f'Attention Matrix A  (T={T} tokens)')
    axes[0].set_xlabel('Key position'); axes[0].set_ylabel('Query position')

    im1 = axes[1].imshow(W_OV, cmap='RdBu_r', aspect='auto')
    plt.colorbar(im1, ax=axes[1], label='Value')
    axes[1].set_title(f'OV Circuit $W_OW_V$ (rank $\\leq d_k={d_k}$)')
    axes[1].set_xlabel('Input dim'); axes[1].set_ylabel('Output dim')
    fig.tight_layout()
    plt.suptitle('Attention: Mixing Weights and Low-Rank OV Circuit',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.show()


---

## Summary

| Section | Key Formula | AI Connection |
|---|---|---|
| OLS | $\hat{\boldsymbol{\beta}}=(X^\top X)^{-1}X^\top\mathbf{y}$ | Linear probing |
| Hat matrix | $H=X(X^\top X)^{-1}X^\top$ | Influence functions |
| Ridge | $(X^\top X+\lambda I)^{-1}X^\top\mathbf{y}$ | Weight decay |
| Lasso | $S_\lambda(z) = \operatorname{sign}(z)\max(|z|-\lambda,0)$ | Sparse autoencoders |
| Bayesian | $\boldsymbol{\mu}_n = \Sigma_n(\Sigma_0^{-1}\boldsymbol{\mu}_0 + \frac{1}{\sigma^2}X^\top\mathbf{y})$ | Uncertainty quantification |
| Logistic | $\nabla\mathcal{L}=\frac{1}{n}X^\top(\mathbf{p}-\mathbf{y})$ | Cross-entropy training |
| SVM | $\min\frac{1}{2}\lVert\mathbf{w}\rVert^2 + C\sum\xi_i$ | Kernel attention |
| Bias-var | $\text{Bias}^2 + \text{Variance} + \sigma^2$ | Double descent |
| LoRA | $W=W_0+BA$, $\operatorname{rank}\leq r$ | PEFT for LLMs |
| NTK | $\Theta(x,x')=\nabla_\theta f(x)^\top\nabla_\theta f(x')$ | Infinite-width theory |

**Next:** `exercises.ipynb` for hands-on implementations.
